# Урок 1.1 — Что такое LLM?

**Модуль 01 «Введение в LLM: с нуля до первого запроса» · Курс «LLM для Python-разработчиков: от RAG до агентов»**

Вы каждый день пользуетесь автокомплитом в IDE. Большая языковая модель (LLM, Large Language Model) — это, по сути, автокомплит, доведённый до абсурда: он «дописывает текст» настолько хорошо, что попутно научился отвечать на вопросы, писать код, переводить и рассуждать. В этом уроке разберём, как это работает изнутри — без математики, на уровне, который нужен инженеру для осознанной работы.

## 🎯 Что вы узнаете

- Что LLM делает на самом деле: предсказание следующего токена (next token prediction) и авторегрессия
- Что такое **токены**, почему модель считает не слова и почему русский текст «дороже» английского
- Как устроен трансформер (transformer) «на пальцах» — и что такое attention
- Что такое **контекстное окно** (context window) и почему это лимит номер один в вашей работе
- Откуда у модели «знания», почему она галлюцинирует и что такое knowledge cutoff

**Время:** ~60–90 минут · **Требования:** Python 3.10+, установленный Jupyter (или VS Code с расширением Jupyter). GPU и платные API **не нужны**.

## Подготовка окружения

Для этого урока нужны две лёгкие библиотеки:

- `tiktoken` — быстрый токенизатор от OpenAI (работает локально, бесплатно);
- `httpx` — HTTP-клиент (понадобится в конце для необязательной демонстрации).

Если вы работаете в отдельном виртуальном окружении (рекомендую), создайте его один раз для всего модуля:

**Windows (PowerShell):**
```powershell
cd C:\Users\rusla\ml_projects\llm-course
python -m venv .venv
.venv\Scripts\Activate.ps1
pip install jupyter tiktoken httpx
```

**Linux / macOS (bash):**
```bash
cd ~/llm-course
python -m venv .venv
source .venv/bin/activate
pip install jupyter tiktoken httpx
```

Либо просто выполните ячейку ниже — `%pip` поставит пакеты в то окружение, где запущен ноутбук.

In [ ]:
from importlib.util import find_spec

if find_spec("tiktoken") is None or find_spec("httpx") is None:
    %pip install -q tiktoken httpx
else:
    print("Зависимости уже установлены")

## 1. LLM — это функция предсказания следующего токена

Уберём всю магию. С точки зрения инженера LLM — это **чистая функция**:

```
f(текст_до_этого_момента) -> распределение вероятностей следующего токена
```

Она принимает последовательность токенов и возвращает для **каждого** токена из своего словаря (обычно ~50–150 тысяч штук) вероятность того, что он будет следующим. Всё. Модель не «думает» в человеческом смысле, не ищет в интернете, не выполняет код — она делает один шаг автокомплита.

Как из одного шага получается целый ответ? **Авторегрессия** (autoregression): выбранный токен дописывается ко входу, и функция вызывается снова. Это обычный цикл `while`:

```
                 ┌──────────────────────────────────────────┐
                 │                                          │
                 ▼                                          │
   ┌─────────┐  ┌──────────────┐  ┌───────────────┐  ┌──────┴──────┐
   │  Промпт  │─▶│ Токенизация │─▶│  Модель (f)   │─▶│ Выбор токена│
   │ (текст)  │  │ текст→числа │  │ → вероятности │  │ и дозапись  │
   └─────────┘  └──────────────┘  └───────────────┘  └──────┬──────┘
                                                            │
                                        стоп-токен или лимит▼
                                                     ┌─────────────┐
                                                     │ Декодирование│
                                                     │ числа→текст  │
                                                     └─────────────┘
```

Аналогия для бэкендера: представьте себе `SELECT next_word, probability FROM language_statistics WHERE context = :everything_so_far` — только вместо таблицы статистики стоит нейросеть, умеющая **обобщать** на контексты, которых никогда не видела.

Из этого простого устройства следуют три важных инженерных вывода:

1. **Генерация последовательна.** Токены рождаются по одному, поэтому длинный ответ = долгий ответ. Отсюда стриминг (streaming) в чатах — токены показывают по мере генерации.
2. **Модель видит только вход.** Никакой «памяти о прошлых диалогах» у самой модели нет — если история чата не передана во вход целиком, модель о ней не знает.
3. **Выход вероятностный.** Один и тот же вход может давать разные выходы — это настройка, а не баг (разберём `temperature` в следующем уроке).

## 2. Токены: валюта мира LLM

Модель не работает с буквами или словами. Она работает с **токенами** (tokens) — кусочками текста из фиксированного словаря. Токеном может быть целое слово (` привет`), часть слова (`програм` + `мист`), знак препинания или даже байт.

Словарь строится алгоритмом **BPE** (Byte Pair Encoding) — идея простая, как сжатие данных: берём огромный корпус текста и жадно склеиваем самые частые пары символов в новые «символы», пока словарь не вырастет до нужного размера (~100k). Частые последовательности («the», « не», «tion») становятся одним токеном, редкие — разваливаются на кусочки.

Почему это важно инженеру:

- **Все лимиты и цены — в токенах.** Контекстное окно 8k, цена $2 за миллион — это про токены, не слова и не символы.
- **Русский текст «дороже» английского.** Словари BPE строились на корпусах, где английского больше, поэтому русские слова чаще дробятся на мелкие куски — тот же смысл занимает в 1.5–3 раза больше токенов.
- **Модель «видит» текст странно.** Знаменитый вопрос «сколько букв r в слове strawberry» ломает модели именно потому, что модель видит не буквы, а токены `str`+`aw`+`berry` — посчитать буквы внутри токена ей трудно.

Посмотрим на токены руками — `tiktoken` использует те же словари, что и модели OpenAI, и отлично подходит для интуиции (у open-source моделей словари свои, но принцип идентичен).

In [ ]:
import tiktoken

# o200k_base — словарь современных моделей OpenAI (gpt-4o и новее)
enc = tiktoken.get_encoding("o200k_base")

text = "Привет! Я backend-разработчик, изучаю LLM."
token_ids = enc.encode(text)

print(f"Текст:  {text!r}")
print(f"Токены (ID): {token_ids}")
print(f"Количество токенов: {len(token_ids)}")
print()

# Посмотрим, как именно текст порезан на кусочки
print("Как модель 'видит' этот текст:")
for tid in token_ids:
    piece = enc.decode([tid])
    print(f"  {tid:>8} -> {piece!r}")

Обратите внимание: пробел обычно «приклеен» к началу следующего токена (`' Я'`, `' backend'`), а русские слова часто разбиты на 2–3 куска. Теперь сравним «стоимость» одинакового смысла на разных языках:

In [ ]:
def count_tokens(text: str, encoding_name: str = "o200k_base") -> int:
    """Возвращает количество токенов в тексте для заданного словаря."""
    enc = tiktoken.get_encoding(encoding_name)
    return len(enc.encode(text))


samples = [
    ("EN", "Please summarize this document and list the key risks."),
    ("RU", "Пожалуйста, сделай краткое резюме этого документа и перечисли ключевые риски."),
    ("EN", "The user cannot log in after the password reset."),
    ("RU", "Пользователь не может войти в систему после сброса пароля."),
    ("Код", "async def get_user(user_id: int) -> User | None: ..."),
]

print(f"{'Язык':<5} {'Символов':>9} {'Токенов':>8}  {'Токенов/символ':>15}")
print("-" * 45)
for lang, s in samples:
    n_chars, n_tokens = len(s), count_tokens(s)
    print(f"{lang:<5} {n_chars:>9} {n_tokens:>8}  {n_tokens / n_chars:>15.2f}")

Типичный результат: на английском ~4 символа на токен, на русском — ~2–3 символа на токен. **Инженерный вывод:** если ваш продукт работает с русским текстом, закладывайте в 1.5–2 раза больше токенов на тот же объём смысла — это напрямую влияет на то, сколько документов влезет в контекст, и на стоимость облачных API.

> 💡 Для open-source моделей (Llama, Qwen, Mistral) токенизаторы другие — они лежат на Hugging Face рядом с моделью (`AutoTokenizer.from_pretrained(...)`). Принцип и порядок цифр те же.

## 3. Как модель выбирает следующий токен

Итак, модель вернула вероятности для всех ~200k токенов словаря. Например, после текста «Столица Франции —» распределение может выглядеть так:

```
' Париж'   ████████████████████████  78%
' город'   ████                      9%
' не'      ██                        4%
' столица' █                         2%
...остальные 200 тысяч токенов делят оставшиеся 7%
```

Дальше есть два принципиально разных пути:

- **Жадный выбор (greedy):** всегда берём самый вероятный токен. Детерминированно, но текст получается скучным и склонным к зацикливанию.
- **Сэмплирование (sampling):** бросаем «взвешенный кубик» — чаще выпадает вероятное, но иногда и что-то менее очевидное. Живой, разнообразный текст, но каждый запуск даёт другой результат.

Насколько «смелым» будет кубик, управляет параметр **temperature** (и его друзья `top_p`, `top_k`) — им посвящена большая часть следующего урока. Пока запомните главное: **недетерминизм LLM — это управляемая настройка сэмплирования, а не загадка**.

## 4. Строим «игрушечную LLM» за 30 строк

Лучший способ прочувствовать принцип «предскажи следующий токен» — написать такую модель самому. Возьмём простейшую статистическую модель — **биграммную** (bigram): для каждого слова посчитаем, какие слова идут после него и как часто. Это честный прадедушка GPT: та же задача, тот же цикл генерации, только вместо нейросети — таблица частот.

Обучим её на маленьком «корпусе» про будни бэкенд-разработчика:

In [ ]:
import random
from collections import Counter, defaultdict

corpus = """
разработчик пишет код и разработчик пишет тесты
разработчик читает логи и находит ошибку в коде
сервис падает и разработчик читает логи
тесты падают и разработчик чинит код
код проходит ревью и сервис едет в продакшен
разработчик пишет документацию и пьёт кофе
сервис едет в продакшен и разработчик пьёт кофе
разработчик находит ошибку и чинит код
логи показывают ошибку и тесты падают
разработчик пьёт кофе и читает логи
"""

# "Токенизация" — в нашей игрушке токен = слово
tokens = corpus.split()
print(f"Корпус: {len(tokens)} токенов, {len(set(tokens))} уникальных")

# "Обучение" — считаем статистику: какое слово идёт после какого
model: dict[str, Counter] = defaultdict(Counter)
for current_word, next_word in zip(tokens, tokens[1:]):
    model[current_word][next_word] += 1

# Посмотрим на "веса" модели для нескольких слов
for word in ["разработчик", "код", "и"]:
    print(f"\nПосле {word!r}: {dict(model[word])}")

«Обучение» заняло миллисекунды: мы просто посчитали частоты. Теперь — **тот самый цикл авторегрессии**, что и в настоящей LLM: предсказали слово → дописали → предсказали следующее.

In [ ]:
def generate(model: dict[str, Counter], start: str, max_tokens: int = 12,
             greedy: bool = False) -> str:
    """Генерирует текст, предсказывая следующее слово по биграммной статистике.

    greedy=True  -> всегда самое частое слово (жадный выбор)
    greedy=False -> случайный выбор пропорционально частоте (сэмплирование)
    """
    result = [start]
    current = start
    for _ in range(max_tokens):
        candidates = model.get(current)
        if not candidates:          # слово ни разу не встречалось в корпусе
            break
        if greedy:
            current = candidates.most_common(1)[0][0]
        else:
            words = list(candidates.keys())
            weights = list(candidates.values())
            current = random.choices(words, weights=weights, k=1)[0]
        result.append(current)
    return " ".join(result)


print("Жадный выбор (детерминированный — всегда одно и то же):")
for _ in range(3):
    print("  ", generate(model, "разработчик", greedy=True))

print("\nСэмплирование (каждый раз новый текст):")
for _ in range(3):
    print("  ", generate(model, "разработчик"))

Запустите ячейку несколько раз. Жадная генерация всегда выдаёт одну и ту же (и быстро зацикливающуюся) фразу, сэмплирование — разные. Вы только что воспроизвели поведение `temperature=0` против `temperature=1` у настоящих LLM.

Добавим и саму **температуру** — она «перевешивает» распределение перед броском кубика: при низкой температуре вероятная опция становится ещё вероятнее, при высокой — шансы выравниваются:

In [ ]:
import math

def sample_with_temperature(candidates: Counter, temperature: float) -> str:
    """Выбирает слово из распределения частот с учётом температуры."""
    words = list(candidates.keys())
    counts = list(candidates.values())
    # log-частоты делим на температуру, затем softmax -> вероятности.
    # Чем ниже температура, тем "острее" распределение.
    logits = [math.log(c) / temperature for c in counts]
    max_logit = max(logits)                      # для численной стабильности
    exps = [math.exp(l - max_logit) for l in logits]
    total = sum(exps)
    probs = [e / total for e in exps]
    return random.choices(words, weights=probs, k=1)[0]


def generate_t(model, start: str, temperature: float, max_tokens: int = 12) -> str:
    result, current = [start], start
    for _ in range(max_tokens):
        candidates = model.get(current)
        if not candidates:
            break
        current = sample_with_temperature(candidates, temperature)
        result.append(current)
    return " ".join(result)


for t in [0.1, 1.0, 3.0]:
    print(f"temperature={t}:")
    for _ in range(2):
        print("  ", generate_t(model, "разработчик", temperature=t))
    print()

При `temperature=0.1` тексты почти одинаковые и «безопасные», при `3.0` — модель несёт что попало. У настоящих LLM ровно та же ручка.

### Чем настоящая LLM отличается от нашей игрушки

| | Биграммная модель | Настоящая LLM |
|---|---|---|
| Контекст для предсказания | 1 предыдущее слово | тысячи–миллионы токенов |
| «Знания» | таблица частот пар слов | миллиарды весов нейросети |
| Обобщение | нет: не видела пару — молчит | да: понимает смысл через эмбеддинги |
| Обучение | подсчёт частот | градиентный спуск на триллионах токенов |

Главная слабость игрушки — она смотрит только на одно слово назад и сравнивает слова **побуквенно** («ошибка» и «баг» для неё чужие). Обе проблемы решает архитектура трансформера — о ней дальше. А представление слов «по смыслу» (эмбеддинги) — это вообще тема целого модуля 02, и она же — фундамент RAG.

## 5. Трансформер «на пальцах»

**Трансформер** (transformer) — архитектура нейросети из статьи «Attention Is All You Need» (2017), на которой построены все современные LLM. Вот путь текста через неё:

```
 "Кот сел на"                          входной текст
      │
      ▼
 [Токенизация]                         текст -> числа: [1359, 4021, 388]
      │
      ▼
 [Эмбеддинги]                          каждый токен -> вектор из тысяч чисел,
      │                                кодирующий его "смысл"
      ▼
 [Слой трансформера]  ─┐
 [Слой трансформера]   │  x N (десятки слоёв; в каждом —
 [Слой трансформера]   │  self-attention + небольшая нейросеть)
 [Слой трансформера]  ─┘
      │
      ▼
 [Вероятности следующего токена]       ' коврик' 34%, ' диван' 21%, ...
```

Сердце каждого слоя — **self-attention** («внимание»). Для каждого токена он отвечает на вопрос: *«на какие другие токены входа мне смотреть, чтобы уточнить свой смысл?»* — и смешивает информацию из релевантных позиций с соответствующими весами.

Аналогия для бэкендера: attention — это как **JOIN с вычисляемой релевантностью**. Обрабатывая токен «его» в фразе «Разработчик перезапустил сервис, потому что **его** убил OOM-killer», модель через attention «подтягивает» информацию из токена «сервис» — и понимает, кого именно убили. Именно неспособность видеть дальние связи убила нашу биграммную игрушку, и именно attention решает эту проблему.

Что ещё полезно знать (без деталей):

- **Параметры** (parameters, «7B», «70B») — это количество чисел-весов в сети. Больше весов = больше «знаний» и лучше рассуждения, но больше памяти и медленнее. 7B ≈ 7 миллиардов весов.
- Attention сравнивает **каждый токен с каждым** — стоимость растёт квадратично от длины входа. Вот почему контекстное окно ограничено, а длинные промпты обрабатываются заметно дольше.
- Внутри модели токены живут как **векторы смысла** — поэтому LLM понимает, что «ошибка» и «баг» — примерно одно и то же, хотя в них нет общих букв.

Этого уровня понимания достаточно для 95% инженерной работы с LLM. Хотите глубже — в конце урока есть ссылки (Karpathy, «The Illustrated Transformer»).

## 6. Контекстное окно — лимит номер один

**Контекстное окно** (context window) — максимальное количество токенов, которое модель может обработать за один вызов: промпт + история диалога + документы + генерируемый ответ, **всё вместе**.

Типичные размеры (порядок, не догма):

| Модель | Окно | Это примерно |
|---|---|---|
| Старые open-source (Llama 2) | 4k | ~5 страниц текста |
| Типичные локальные модели (Qwen 2.5, Llama 3.x) | 32k–128k | ~40–160 страниц |
| Флагманские облачные | 200k–1M+ | книга или несколько |

Три вещи, которые обязан помнить инженер:

1. **Окно ≠ память.** HTTP-запросы к LLM — stateless, как и положено. «Память» чат-бота — это то, что ваш код передал в промпт. История растёт с каждым сообщением → рано или поздно перестанет влезать → нужно резать, суммировать или искать релевантное (спойлер: RAG, модуль 03).
2. **Больше контекста ≠ лучше.** Эффект «**lost in the middle**»: модели хуже используют информацию из середины длинного контекста, чем из начала и конца. Запихнуть в промпт всю базу знаний — плохая стратегия, даже если она влезает.
3. **Контекст стоит денег и времени.** В облаке платят за каждый входной токен; локально длинный контекст ест RAM и замедляет ответ.

Посчитаем на практике, как быстро «съедается» окно в обычном чат-боте:

In [ ]:
SYSTEM_PROMPT = (
    "Ты — вежливый ассистент службы поддержки интернет-магазина. "
    "Отвечай кратко, по делу, на русском языке. Если не знаешь ответа, "
    "честно скажи об этом и предложи связаться с оператором."
)

# Смоделируем диалог: у пользователя проблема с заказом
user_msg = "Здравствуйте! Я оформил заказ №48291 три дня назад, но он всё ещё не отправлен. Подскажите, что происходит?"
bot_msg = "Здравствуйте! Проверяю ваш заказ №48291... Вижу, что он задержан на складе из-за отсутствия одной позиции. Ожидаемая отправка — завтра. Приносим извинения за задержку!"

CONTEXT_WINDOW = 8192   # окно типичной небольшой локальной модели
RESERVED_FOR_ANSWER = 512  # резервируем место под генерацию ответа

history_tokens = count_tokens(SYSTEM_PROMPT)
turn = 0
print(f"Системный промпт: {history_tokens} токенов\n")
print(f"{'Ход диалога':>12} {'Токенов в истории':>18} {'Осталось места':>15}")
print("-" * 50)

while True:
    turn += 1
    # каждый ход добавляет вопрос пользователя и ответ бота
    history_tokens += count_tokens(user_msg) + count_tokens(bot_msg)
    free = CONTEXT_WINDOW - RESERVED_FOR_ANSWER - history_tokens
    if turn <= 5 or free <= 0:
        print(f"{turn:>12} {history_tokens:>18} {free:>15}")
    if free <= 0:
        print(f"\nОкно переполнено на ходе №{turn}: историю пора резать или суммировать!")
        break

Реальный чат с не самыми длинными сообщениями упирается в окно 8k за несколько десятков ходов. А если добавить в промпт пару документов по 3–5k токенов — за 2–3 хода. Управление контекстом — одна из главных постоянных задач AI-инженера.

## 7. Откуда у модели «знания» — и почему она галлюцинирует

Откуда веса? Модель проходит **предобучение** (pretraining): ей показывают триллионы токенов текста (веб, книги, код) и на каждом учат предсказывать следующий токен. Ошиблась — веса чуть-чуть подправляются (градиентный спуск). После месяцев такого обучения на тысячах GPU веса кодируют колоссальную статистику языка и фактов — это и есть «знания».

Отсюда три фундаментальных следствия:

**1. Knowledge cutoff (дата среза знаний).** Модель знает только то, что было в обучающих данных. Вышла новая версия FastAPI, поменялось API библиотеки — модель об этом не знает и уверенно расскажет про старую версию. Лекарство: передавать актуальную информацию в контекст (это буквально определение RAG — модуль 03).

**2. Галлюцинации (hallucinations).** Модель оптимизировали генерировать **правдоподобный** текст, а не **истинный**. Если знаний не хватает, она не вернёт `null` — она сгенерирует наиболее правдоподобное продолжение: несуществующую функцию в библиотеке, выдуманную статью, фейковый номер дела. Это не баг, а прямое следствие устройства. Инженерные лекарства: grounding через RAG, проверка ответов, правильные промпты — всё это впереди в курсе.

**3. Модель после предобучения — просто «дописыватель текста»** (base-модель). Чтобы она вела диалог и следовала инструкциям, её дополнительно дообучают (instruction tuning, RLHF). Разница между base- и instruct-моделями — критически важная для практики тема урока 1.6.

> ⚙️ Важно для самооценки: всё это — **не** причина не использовать LLM. Это причина использовать их **инженерно**: с проверками, с подачей фактов в контекст, с тестами. Ровно этому и посвящён курс.

## 8. (Опционально) Заглянем вперёд: живой запрос к локальной LLM

В уроке 1.5 мы установим **Ollama** — движок для запуска LLM локально. Если вам не терпится и Ollama уже установлена (`ollama pull qwen2.5:3b`), ячейка ниже отправит первый запрос. Если нет — ячейка вежливо об этом сообщит, и это нормально: вернётесь сюда после урока 1.5.

In [ ]:
import httpx

OLLAMA_URL = "http://localhost:11434"

try:
    # Проверим, жив ли демон Ollama
    httpx.get(OLLAMA_URL, timeout=2)
except (httpx.ConnectError, httpx.TimeoutException):
    print("Ollama не запущена — пропустите эту ячейку, мы установим её в уроке 1.5")
else:
    response = httpx.post(
        f"{OLLAMA_URL}/api/generate",
        json={
            "model": "qwen2.5:3b",       # или любая модель из `ollama list`
            "prompt": "Объясни одним предложением, что такое токен в LLM.",
            "stream": False,
        },
        timeout=120,
    )
    response.raise_for_status()
    data = response.json()
    print("Ответ модели:", data["response"].strip())
    print(f"\nТокенов в промпте: {data.get('prompt_eval_count', '?')}, "
          f"в ответе: {data.get('eval_count', '?')}")

## ⚠️ Частые ошибки

1. **Считать лимиты в словах или символах.** Все лимиты — в токенах, и для русского текста токенов в ~2 раза больше, чем кажется. Всегда считайте токенизатором, а не «на глаз».
2. **Ждать от LLM детерминизма.** Даже при `temperature=0` разные версии модели, разные бэкенды и особенности вычислений с плавающей точкой могут давать разные ответы. Не стройте логику на точном совпадении строк — парсите и валидируйте.
3. **Думать, что модель «помнит» прошлые запросы.** Каждый вызов — с чистого листа. Память диалога — ответственность вашего кода: собрать историю и передать её в промпт.
4. **Верить уверенному тону.** Модель одинаково уверенно сообщает правду и галлюцинацию. Уверенность формулировки — не сигнал достоверности; проверяйте факты, особенно имена функций, версии и цифры.

## Упражнения

**Упражнение 1 — калькулятор стоимости промпта.**
Напишите функцию `estimate_cost(text, price_per_million=2.0)`, которая считает токены в тексте (словарь `o200k_base`) и возвращает стоимость его отправки в модель по цене `price_per_million` долларов за миллион входных токенов. Посчитайте стоимость отправки файла с текстом ~10 000 русских слов (сгенерируйте такой текст программно, например повторением абзаца).

**Упражнение 2 — триграммная модель.**
Наша игрушечная модель смотрела на 1 слово назад. Сделайте версию, которая смотрит на **2 слова назад** (ключ таблицы — кортеж из двух слов). Сравните связность текста биграммной и триграммной версии на том же корпусе. Что случилось с разнообразием и почему? Как это связано с размером контекста у настоящих LLM?

**Упражнение 3 — переполнение окна (без кода или с кодом — как удобнее).**
Ваш чат-бот работает на модели с окном 32k токенов. Системный промпт — 800 токенов, к каждому запросу прикладывается документация на 6 000 токенов, средний вопрос пользователя — 60 токенов, средний ответ — 350 токенов, под генерацию резервируется 1 000 токенов. Сколько ходов диалога выдержит бот, если история хранится целиком? А если хранить только последние 4 хода?

Решения — в конце ноутбука, но сначала попробуйте сами.

In [ ]:
# Упражнение 1 — ваш код здесь:

def estimate_cost(text: str, price_per_million: float = 2.0) -> float:
    """Стоимость отправки текста в модель, USD."""
    n_tokens = count_tokens(text)
    return n_tokens / 1_000_000 * price_per_million


paragraph = (
    "Микросервис обрабатывает входящие запросы, проверяет права доступа, "
    "обращается к базе данных и возвращает ответ в формате JSON. "
    "При ошибке запись попадает в лог и метрики увеличиваются. "
)
# 26 слов в абзаце -> повторим до ~10 000 слов
big_text = paragraph * 385
n_words = len(big_text.split())
n_tokens = count_tokens(big_text)

print(f"Слов: {n_words}, токенов: {n_tokens} (~{n_tokens/n_words:.1f} токена на слово)")
print(f"Стоимость при $2/1M входных токенов: ${estimate_cost(big_text):.4f}")
print(f"А 1000 таких запросов в день за месяц: ${estimate_cost(big_text) * 1000 * 30:.2f}")

In [ ]:
# Упражнение 2 — ваш код здесь:

trigram_model: dict[tuple[str, str], Counter] = defaultdict(Counter)
for w1, w2, w3 in zip(tokens, tokens[1:], tokens[2:]):
    trigram_model[(w1, w2)][w3] += 1


def generate_trigram(start_pair: tuple[str, str], max_tokens: int = 12) -> str:
    result = list(start_pair)
    current = start_pair
    for _ in range(max_tokens):
        candidates = trigram_model.get(current)
        if not candidates:
            break
        words = list(candidates.keys())
        weights = list(candidates.values())
        next_word = random.choices(words, weights=weights, k=1)[0]
        result.append(next_word)
        current = (current[1], next_word)   # окно контекста "сдвигается"
    return " ".join(result)


print("Биграммы (контекст = 1 слово):")
for _ in range(3):
    print("  ", generate(model, "разработчик"))

print("\nТриграммы (контекст = 2 слова):")
for _ in range(3):
    print("  ", generate_trigram(("разработчик", "пишет")))

# Наблюдение: триграммы связнее (почти дословно цитируют корпус), но менее
# разнообразны — вариантов продолжения для пары слов в маленьком корпусе мало.
# Это фундаментальный компромисс: длиннее контекст -> нужнее больше данных.
# LLM решают его за счёт триллионов токенов обучения и способности обобщать.

## Мини-квиз

Проверьте себя (ответы — в следующей ячейке, не подглядывайте раньше времени):

**1.** Что возвращает LLM за один вызов «внутренней функции»?
   a) готовый ответ целиком  b) распределение вероятностей следующего токена  c) самый вероятный следующий токен  d) вектор смысла текста

**2.** Почему русский текст обычно занимает больше токенов, чем английский того же смысла?
   a) кириллица кодируется двумя байтами  b) в словаре BPE меньше длинных русских кусков, слова дробятся мельче  c) русские слова длиннее  d) это миф, разницы нет

**3.** Что из перечисленного — **контекстное окно**?
   a) память модели о прошлых диалогах  b) максимум токенов на вход и выход одного вызова  c) размер словаря токенизатора  d) объём RAM для запуска модели

**4.** Чат-бот на LLM «помнит» ваше имя из начала разговора, потому что…
   a) модель дообучилась на вашем диалоге  b) имя сохранилось в весах  c) код бота передаёт всю историю диалога в каждый новый запрос  d) модель хранит сессию по IP

**5.** Модель уверенно назвала функцию `fastapi.magic_cache()`, которой не существует. Это…
   a) баг конкретной модели, надо обновить  b) галлюцинация — закономерное следствие обучения на правдоподобие  c) устаревшие знания: функция была в старых версиях  d) результат высокой температуры, при 0 такого не бывает

<details>
<summary><b>Ответы на квиз (нажмите, чтобы раскрыть)</b></summary>

**1 — b.** Модель возвращает вероятности для всех токенов словаря; выбор конкретного токена (жадно или сэмплированием) — отдельный шаг, а полный ответ собирается циклом авторегрессии.

**2 — b.** Словарь BPE строился на корпусе с преобладанием английского: частые английские последовательности стали длинными токенами, а русские слова дробятся на 2–4 куска. Байты кириллицы (вариант a) играют роль лишь косвенно.

**3 — b.** Окно — суммарный лимит токенов одного вызова: системный промпт + история + документы + генерируемый ответ. Память (a) — иллюзия, создаваемая вашим кодом.

**4 — c.** HTTP-вызовы модели stateless. «Память» — это история, которую ваш код прикладывает к каждому запросу (и которая ест контекстное окно).

**5 — b.** Галлюцинация: модель генерирует правдоподобное продолжение, а несуществующее-но-логичное имя функции правдоподобно. Бывает и при `temperature=0` — низкая температура снижает разнообразие, но не добавляет знаний.

</details>

## Решения упражнений

In [ ]:
# Решение упражнения 1

def estimate_cost(text: str, price_per_million: float = 2.0) -> float:
    """Стоимость отправки текста в модель, USD."""
    n_tokens = count_tokens(text)
    return n_tokens / 1_000_000 * price_per_million


paragraph = (
    "Микросервис обрабатывает входящие запросы, проверяет права доступа, "
    "обращается к базе данных и возвращает ответ в формате JSON. "
    "При ошибке запись попадает в лог и метрики увеличиваются. "
)
# 26 слов в абзаце -> повторим до ~10 000 слов
big_text = paragraph * 385
n_words = len(big_text.split())
n_tokens = count_tokens(big_text)

print(f"Слов: {n_words}, токенов: {n_tokens} (~{n_tokens/n_words:.1f} токена на слово)")
print(f"Стоимость при $2/1M входных токенов: ${estimate_cost(big_text):.4f}")
print(f"А 1000 таких запросов в день за месяц: ${estimate_cost(big_text) * 1000 * 30:.2f}")
# Вывод: копейки за один запрос легко превращаются в сотни долларов на потоке.

In [ ]:
# Решение упражнения 2

trigram_model: dict[tuple[str, str], Counter] = defaultdict(Counter)
for w1, w2, w3 in zip(tokens, tokens[1:], tokens[2:]):
    trigram_model[(w1, w2)][w3] += 1


def generate_trigram(start_pair: tuple[str, str], max_tokens: int = 12) -> str:
    result = list(start_pair)
    current = start_pair
    for _ in range(max_tokens):
        candidates = trigram_model.get(current)
        if not candidates:
            break
        words = list(candidates.keys())
        weights = list(candidates.values())
        next_word = random.choices(words, weights=weights, k=1)[0]
        result.append(next_word)
        current = (current[1], next_word)   # окно контекста "сдвигается"
    return " ".join(result)


print("Биграммы (контекст = 1 слово):")
for _ in range(3):
    print("  ", generate(model, "разработчик"))

print("\nТриграммы (контекст = 2 слова):")
for _ in range(3):
    print("  ", generate_trigram(("разработчик", "пишет")))

# Наблюдение: триграммы связнее (почти дословно цитируют корпус), но менее
# разнообразны — вариантов продолжения для пары слов в маленьком корпусе мало.
# Это фундаментальный компромисс: длиннее контекст -> нужнее больше данных.
# LLM решают его за счёт триллионов токенов обучения и способности обобщать.

**Решение упражнения 3 (расчёт).**

Постоянная часть запроса: системный промпт 800 + документация 6 000 + резерв на ответ 1 000 = **7 800 токенов**. Свободно под историю: 32 768 − 7 800 = **24 968 токенов**. Один ход диалога = вопрос 60 + ответ 350 = **410 токенов**.

- История целиком: 24 968 / 410 ≈ **60 ходов** — дальше окно переполнится.
- Только последние 4 хода: история никогда не превышает 4 × 410 = 1 640 токенов — бот работает **бесконечно**, но «забывает» всё старше 4 ходов.

Инженерный компромисс между «помнит всё, но недолго живёт» и «живёт вечно, но забывает» решают гибридные схемы: суммаризация старой истории, выборка релевантных кусков (RAG) — об этом в модулях 03–04.

## Итоги

- LLM — функция «токены → вероятности следующего токена»; ответы собираются циклом авторегрессии, токен за токеном.
- Токены — универсальная валюта: в них меряются лимиты, цены и скорость; русский текст в ~2 раза «дороже» английского.
- Трансформер и attention позволяют модели учитывать связи между далёкими токенами — то, чего лишена простая статистика.
- Контекстное окно — жёсткий лимит на «промпт + история + документы + ответ»; окно ≠ память, память — забота вашего кода.
- Знания модели заморожены на момент обучения; галлюцинации — следствие оптимизации на правдоподобие. Лекарства — инженерные, и им посвящён весь курс.

**Дальше:** урок 1.2 «Осознанное взаимодействие с LLM» — учимся управлять генерацией: системные промпты, `temperature`, `top_p`, `max_tokens`, стоп-последовательности и первые правила безопасности.

## 📚 Полезные ссылки

- [tiktoken](https://github.com/openai/tiktoken) — токенизатор, который мы использовали
- [Tiktokenizer](https://tiktokenizer.vercel.app/) — интерактивная визуализация токенизации в браузере
- [The Illustrated Transformer](https://jalammar.github.io/illustrated-transformer/) — лучшее визуальное объяснение трансформера (англ.)
- [Andrej Karpathy — Intro to Large Language Models](https://www.youtube.com/watch?v=zjkBMFhNj_g) — часовой обзор LLM «для людей» (англ.)
- [Ollama](https://ollama.com/) — установим в уроке 1.5
- «Attention Is All You Need» ([arXiv:1706.03762](https://arxiv.org/abs/1706.03762)) — та самая статья 2017 года (читать не обязательно 🙂)